<img src="https://images.squarespace-cdn.com/content/v1/67351b6c456f5a62d1c1bca2/17a93d03-0cae-49bf-b152-8dfd0cdb9d7d/Quantum+Rings+Logo+Main+White.png?format=300w" alt="Quantum Rings" width="150" align="right"/>

# Shor's Algorithm
Shor's algorithm is a groundbreaking quantum algorithm that solves the problem of integer factorization exponentially faster than the best-known classical algorithms. It can efficiently find the prime factors of a large integer, which has significant implications for cryptography, particularly in breaking widely-used encryption systems like RSA. The core advantage of Shor's algorithm lies in its ability to leverage quantum properties such as superposition and entanglement to perform calculations that would be infeasible on classical computers. This makes it one of the most famous examples highlighting the potential power of quantum computing.

Learn Shors through the complete documentation here:
https://portal.quantumrings.com/doc/Shors.html

Now, let's look at a simple implementation of Shor's algorithm:

In [ ]:
%%capture
%pip install QuantumRingsLib matplotlib numpy

In [ ]:
import QuantumRingsLib
from QuantumRingsLib import QuantumRegister, AncillaRegister, ClassicalRegister, QuantumCircuit
from QuantumRingsLib import QuantumRingsProvider
from QuantumRingsLib import job_monitor
from QuantumRingsLib import JobStatus
from matplotlib import pyplot as plt
import numpy as np
import math

In [ ]:
provider = QuantumRingsProvider(token="YOUR_TOKEN_HERE", name="YOUR_EMAIL_ADDRESS")
backend = provider.get_backend("scarlet_quantum_rings")
shots = 1024

provider.active_account()

In [ ]:
def iqft_cct(qc, b, n):
    """
    The inverse QFT circuit

    Args:

        qc (QuantumCircuit):
                The quantum circuit

        b (QuantumRegister):
                The target register

        n (int):
                The number of qubits in the registers to use

    Returns:
        None

    """

    for i in range (n):
        for j in range (1, i+1):
            # for inverse transform, we have to use negative angles
            qc.cu1(  -math.pi / 2** ( i -j + 1 ), b[j - 1], b[i])
        # the H transform should be done after the rotations
        qc.h(b[i])
    qc.barrier()
    return

def plot_histogram (counts, title=""):
    """
    Plots the histogram of the counts

    Args:

        counts (dict):
            The dictionary containing the counts of states

        titles (str):
            A title for the graph.

    Returns:
        None

    """
    fig, ax = plt.subplots(figsize =(10, 7))
    plt.xlabel("States")
    plt.ylabel("Counts")
    mylist = [key for key, val in counts.items() for _ in range(val)]

    unique, inverse = np.unique(mylist, return_inverse=True)
    bin_counts = np.bincount(inverse)

    plt.bar(unique, bin_counts)

    maxFreq = max(counts.values())
    plt.ylim(ymax=np.ceil(maxFreq / 10) * 10 if maxFreq % 10 else maxFreq + 10)
    # Show plot
    plt.title(title)
    plt.show()
    return

### Execute the circuit

In [ ]:
# Shor’s algorithm to factorize 15 using 7^x mod 15.
numberofqubits = 7
shots = 1024

q = QuantumRegister(numberofqubits , 'q')
c = ClassicalRegister(4 , 'c')
qc = QuantumCircuit(q, c)

# Initialize source and target registers
qc.h(0)
qc.h(1)
qc.h(2)
qc.x(6)
qc.barrier()

# Modular exponentiation 7^x mod 15
qc.cx(q[2],q[4] )
qc.cx(q[2],q[5] )
qc.cx(q[6],q[4] )
qc.ccx(q[1],q[5],q[3] )
qc.cx(q[3],q[5] )
qc.ccx(q[1],q[4],q[6] )
qc.cx(q[6],q[4] ) #
qc.barrier()

# IQFT. Refer to implementation from earlier examples
iqft_cct (qc, q, 3)

# Measure
qc.measure(q[0], c[0])
qc.measure(q[1], c[1])
qc.measure(q[2], c[2])

# Execute the circuit
job = backend.run(qc, shots)
job_monitor(job)
result = job.result()
counts = result.get_counts()

# Print the counts dictionary
print("Measurement Results (Counts):")
print(counts)

#clean up
del q, c, qc
del result
del job

#visualize
plot_histogram(counts)